# ResNet sur CIFAR-10 (10%)

ResNet signifie **Residual Network**. L'idee principale est simple : le reseau garde une partie de l'information telle quelle grace a des raccourcis, puis apprend seulement ce qu'il faut ajouter ou corriger.


In [ ]:
%pip install --quiet torch transformers datasets accelerate matplotlib scikit-learn

In [ ]:
from pathlib import Path
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import ResNetConfig, ResNetForImageClassification, Trainer, TrainingArguments

# ====================== CONFIGURATION ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ====================== CHEMINS ======================
CIFAR_DIR = Path('./cifar-10-batches-py')

if not CIFAR_DIR.exists():
    raise FileNotFoundError(f'Missing CIFAR-10 directory: {CIFAR_DIR}')

OUTPUT_DIR = Path('./outputs/cifar10_resnet')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ====================== PARAMETRES ======================
SUBSET_FRACTION = 0.25
VAL_FRACTION = 0.20
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 5e-4
FINAL_EPOCHS = 30

print('CUDA available:', torch.cuda.is_available())
print('CIFAR dir:', CIFAR_DIR.resolve())
print('Output dir:', OUTPUT_DIR)

In [ ]:
train_batches = []
train_labels = []

for batch_id in range(1, 6):
    with (CIFAR_DIR / f'data_batch_{batch_id}').open('rb') as handle:
        batch = pickle.load(handle, encoding='latin1')
    train_batches.append(batch['data'])
    train_labels.extend(batch['labels'])

with (CIFAR_DIR / 'test_batch').open('rb') as handle:
    test_batch = pickle.load(handle, encoding='latin1')

with (CIFAR_DIR / 'batches.meta').open('rb') as handle:
    meta = pickle.load(handle, encoding='latin1')

class_names = meta['label_names']
id2label = {idx: label for idx, label in enumerate(class_names)}
label2id = {label: idx for idx, label in id2label.items()}

x_train_full = np.concatenate(train_batches).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
y_train_full = np.asarray(train_labels)
x_test_full = test_batch['data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
y_test_full = np.asarray(test_batch['labels'])

train_subset_indices, _ = train_test_split(
    np.arange(len(y_train_full)),
    train_size=SUBSET_FRACTION,
    stratify=y_train_full,
    random_state=SEED,
)

train_indices, val_indices = train_test_split(
    train_subset_indices,
    test_size=VAL_FRACTION,
    stratify=y_train_full[train_subset_indices],
    random_state=SEED,
)

test_indices, _ = train_test_split(
    np.arange(len(y_test_full)),
    train_size=SUBSET_FRACTION,
    stratify=y_test_full,
    random_state=SEED,
)

print('Classes:', class_names)
print('Full train images:', len(x_train_full))
print('Full test images:', len(x_test_full))
print('Train images used:', len(train_indices))
print('Validation images used:', len(val_indices))
print('Test images used:', len(test_indices))

In [ ]:
mean = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32).reshape(1, 1, 1, 3)
std = np.array([0.2470, 0.2435, 0.2616], dtype=np.float32).reshape(1, 1, 1, 3)

x_train = ((x_train_full[train_indices].astype(np.float32) / 255.0) - mean) / std
x_val = ((x_train_full[val_indices].astype(np.float32) / 255.0) - mean) / std
x_test = ((x_test_full[test_indices].astype(np.float32) / 255.0) - mean) / std

x_train = np.transpose(x_train, (0, 3, 1, 2))
x_val = np.transpose(x_val, (0, 3, 1, 2))
x_test = np.transpose(x_test, (0, 3, 1, 2))

train_dataset = Dataset.from_dict({
    'pixel_values': x_train,
    'labels': y_train_full[train_indices],
})
val_dataset = Dataset.from_dict({
    'pixel_values': x_val,
    'labels': y_train_full[val_indices],
})
test_dataset = Dataset.from_dict({
    'pixel_values': x_test,
    'labels': y_test_full[test_indices],
})

train_dataset.set_format('torch')
val_dataset.set_format('torch')
test_dataset.set_format('torch')

print(train_dataset[0]['pixel_values'].shape)
print(train_dataset[0]['labels'])

In [ ]:
config = ResNetConfig(
    num_channels=3,
    num_labels=len(class_names),
    id2label=id2label,
    label2id=label2id,
    depths=[2, 2, 2, 2],
    hidden_sizes=[64, 128, 256, 512],
    embedding_size=64,
)

model = ResNetForImageClassification(config)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'final'),
    num_train_epochs=FINAL_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine"
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    seed=SEED,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print('Ready to train ResNet')

In [ ]:
train_result = trainer.train()
val_results = trainer.evaluate()
test_output = trainer.predict(test_dataset)

y_true = test_output.label_ids
y_pred = np.argmax(test_output.predictions, axis=1)
test_accuracy = accuracy_score(y_true, y_pred)

print('Validation results:', val_results)
print('Test loss:', test_output.metrics['test_loss'])
print(f'Test accuracy: {test_accuracy:.4f}')

In [ ]:
trainer.save_model(str(OUTPUT_DIR / 'final_model'))
print('Model saved to:', OUTPUT_DIR / 'final_model')

In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap='Blues')
plt.title('Matrice de confusion')
plt.colorbar()
plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right')
plt.yticks(range(len(class_names)), class_names)
plt.xlabel('Prediction')
plt.ylabel('Classe reelle')
plt.tight_layout()
plt.show()

In [ ]:
sample_indices = np.random.default_rng(SEED).choice(len(test_indices), size=12, replace=False)
sample_images = x_test_full[test_indices][sample_indices]
sample_true = y_test_full[test_indices][sample_indices]
sample_pred = y_pred[sample_indices]

plt.figure(figsize=(12, 6))
for plot_idx, (image, true_label, pred_label) in enumerate(zip(sample_images, sample_true, sample_pred), start=1):
    color = 'green' if true_label == pred_label else 'red'
    plt.subplot(3, 4, plot_idx)
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"{class_names[pred_label]}\nreal: {class_names[true_label]}", color=color)

plt.tight_layout()
plt.show()